In [82]:
import argparse
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [83]:
DATA_DIR = Path('C:\march_mania\march-machine-learning-mania-2026')     # ← folder containing all CSV files
OUT_DIR  = Path("./output")   # ← where charts will be saved
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [84]:
DATA = DATA_DIR

In [85]:
teams           = pd.read_csv(DATA / "MTeams.csv")
seasons         = pd.read_csv(DATA / "MSeasons.csv")
team_spellings  = pd.read_csv(DATA / "MTeamSpellings.csv")

# ── Tournament structure ──────────────────────────────────────────
seeds           = pd.read_csv(DATA / "MNCAATourneySeeds.csv")
tourney_slots   = pd.read_csv(DATA / "MNCAATourneySlots.csv")
seed_rnd_slots  = pd.read_csv(DATA / "MNCAATourneySeedRoundSlots.csv")

# ── Game results ──────────────────────────────────────────────────
tourney_compact = pd.read_csv(DATA / "MNCAATourneyCompactResults.csv")
tourney_detail  = pd.read_csv(DATA / "MNCAATourneyDetailedResults.csv")
reg_compact     = pd.read_csv(DATA / "MRegularSeasonCompactResults.csv")
reg_detail      = pd.read_csv(DATA / "MRegularSeasonDetailedResults.csv")

# ── Conference & coaching data ────────────────────────────────────
conferences     = pd.read_csv(DATA / "Conferences.csv")
team_conf       = pd.read_csv(DATA / "MTeamConferences.csv")
conf_games      = pd.read_csv(DATA / "MConferenceTourneyGames.csv")
coaches         = pd.read_csv(DATA / "MTeamCoaches.csv")

# ── Secondary tournaments & geography ────────────────────────────
secondary_res   = pd.read_csv(DATA / "MSecondaryTourneyCompactResults.csv")
secondary_teams = pd.read_csv(DATA / "MSecondaryTourneyTeams.csv")
cities          = pd.read_csv(DATA / "Cities.csv")
game_cities     = pd.read_csv(DATA / "MGameCities.csv")

In [86]:
secondary_res.sample(5)

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,SecondaryTourney
57,1986,144,1256,64,1344,63,A,0,NIT
1860,2025,150,1416,88,1153,80,N,0,CBC
1196,2014,135,1229,77,1287,67,H,0,CBI
813,2008,142,1409,73,1222,69,H,0,CBI
119,1988,140,1433,93,1379,89,H,0,NIT


In [87]:
secondary_teams.sample(5)

,Season,SecondaryTourney,TeamID
720,2006,NIT,1257
1629,2019,NIT,1313
1635,2019,NIT,1395
513,2001,NIT,1120
1195,2014,CBI,1392


In [88]:
cities.sample(5)

,CityID,City,State
419,4442,Toronto,ON
359,4361,Villanova,PA
492,4515,Palm Springs,CA
65,4066,Charlotte,NC
315,4317,Smithfield,RI


In [89]:
game_cities.sample(5)

,Season,DayNum,WTeamID,LTeamID,CRType,CityID
11139,2012,18,1210,1433,Regular,4064
57001,2020,59,1398,1183,Regular,4233
82744,2025,41,1297,1313,Regular,4153
63009,2021,104,1304,1336,Regular,4355
35522,2016,77,1235,1328,Regular,4006


In [90]:
reg_compact.sample(5)

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
10363,1987,105,1212,77,1108,67,H,0
148174,2017,77,1126,82,1197,76,A,0
52942,1998,43,1291,69,1162,42,H,0
75647,2003,61,1173,92,1266,85,H,1
53561,1998,64,1437,83,1130,76,A,0


In [91]:
tourney_detail.sample(5)

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
753,2014,138,1276,79,1400,65,N,0,24,54,...,11,15,16,21,20,14,9,3,3,16
185,2005,145,1228,90,1112,89,N,1,32,71,...,18,18,21,12,25,21,17,11,8,10
847,2016,134,1195,96,1192,65,N,0,34,57,...,23,11,18,9,24,16,9,9,1,26
253,2006,152,1196,73,1206,58,N,0,23,53,...,11,10,14,11,16,6,11,8,2,15
406,2009,137,1257,74,1287,54,N,0,29,50,...,11,10,14,13,14,12,20,4,1,10


In [92]:
reg_detail.sample(5)

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
57091,2014,55,1294,84,1315,66,H,0,29,48,...,21,13,23,16,9,10,7,9,2,14
968,2003,42,1207,84,1313,48,H,0,26,62,...,21,9,17,15,22,10,28,15,5,29
25282,2008,46,1248,76,1291,72,A,0,27,49,...,26,17,22,10,17,14,20,8,4,18
97099,2022,16,1102,61,1411,57,H,0,21,47,...,12,15,19,10,23,7,16,6,1,21
61936,2015,41,1385,74,1200,53,H,0,27,56,...,22,10,17,11,23,11,20,8,4,21


In [93]:
len(reg_detail.columns)

34

In [94]:
w_cols = {
    'Season': 'Season', 'WTeamID': 'TeamID',
    'WScore': 'Score', 'LScore': 'OppScore',
    'WFGM': 'FGM', 'WFGA': 'FGA',
    'WFGM3': 'FGM3', 'WFGA3': 'FGA3',
    'WFTM': 'FTM', 'WFTA': 'FTA',
    'WOR': 'OR', 'WDR': 'DR',
    'WAst': 'Ast', 'WTO': 'TO',
    'WStl': 'Stl', 'WBlk': 'Blk', 'WPF': 'PF',
    'LFGM': 'Opp_FGM', 'LFGA': 'Opp_FGA',
    'LFGM3': 'Opp_FGM3', 'LFGA3': 'Opp_FGA3',
    'LOR': 'Opp_OR', 'LDR': 'Opp_DR',
    'LTO': 'Opp_TO'
}
winners = reg_detail[list(w_cols.keys())].rename(columns=w_cols)
winners['Win'] = 1

In [95]:
l_cols = {
    'Season': 'Season', 'LTeamID': 'TeamID',
    'LScore': 'Score', 'WScore': 'OppScore',
    'LFGM': 'FGM', 'LFGA': 'FGA',
    'LFGM3': 'FGM3', 'LFGA3': 'FGA3',
    'LFTM': 'FTM', 'LFTA': 'FTA',
    'LOR': 'OR', 'LDR': 'DR',
    'LAst': 'Ast', 'LTO': 'TO',
    'LStl': 'Stl', 'LBlk': 'Blk', 'LPF': 'PF',
    'WFGM': 'Opp_FGM', 'WFGA': 'Opp_FGA',
    'WFGM3': 'Opp_FGM3', 'WFGA3': 'Opp_FGA3',
    'WOR': 'Opp_OR', 'WDR': 'Opp_DR',
    'WTO': 'Opp_TO'
}
losers = reg_detail[list(l_cols.keys())].rename(columns=l_cols)
losers['Win'] = 0

In [ ]:
losers

In [ ]:
winners.shape

In [ ]:
all_games = pd.concat([winners,losers],ignore_index = True)
all_games.sample(5)

In [96]:
all_games.shape

(245550, 25)

In [97]:
all_games.isnull().sum()

Season      0
TeamID      0
Score       0
OppScore    0
FGM         0
FGA         0
FGM3        0
FGA3        0
FTM         0
FTA         0
OR          0
DR          0
Ast         0
TO          0
Stl         0
Blk         0
PF          0
Opp_FGM     0
Opp_FGA     0
Opp_FGM3    0
Opp_FGA3    0
Opp_OR      0
Opp_DR      0
Opp_TO      0
Win         0
dtype: int64

In [98]:
all_games.describe()

,Season,TeamID,Score,OppScore,FGM,FGA,FGM3,FGA3,FTM,FTA,...,Blk,PF,Opp_FGM,Opp_FGA,Opp_FGM3,Opp_FGA3,Opp_OR,Opp_DR,Opp_TO,Win
count,245550.000000,245550.000000,245550.000000,245550.000000,245550.000000,245550.000000,245550.000000,245550.000000,245550.000000,245550.000000,...,245550.000000,245550.000000,245550.000000,245550.000000,245550.000000,245550.000000,245550.000000,245550.000000,245550.000000,245550.000000
mean,2014.622464,1285.944366,70.070841,70.070841,24.658265,56.356832,6.820867,19.879352,13.933443,19.884231,...,3.326064,18.206956,24.658265,56.356832,6.820867,19.879352,10.355471,23.586777,13.147954,0.500000
std,6.784944,105.263475,12.503686,12.503686,4.874815,7.502384,3.036805,6.108928,6.087782,7.890220,...,2.268443,4.471752,4.874815,7.502384,3.036805,6.108928,4.165299,5.107574,4.255828,0.500001
min,2003.000000,1101.000000,20.000000,20.000000,6.000000,26.000000,0.000000,1.000000,0.000000,0.000000,...,0.000000,3.000000,6.000000,26.000000,0.000000,1.000000,0.000000,4.000000,0.000000,0.000000
25%,2009.000000,1195.000000,62.000000,62.000000,21.000000,51.000000,5.000000,16.000000,9.000000,14.000000,...,2.000000,15.000000,21.000000,51.000000,5.000000,16.000000,7.000000,20.000000,10.000000,0.000000
50%,2015.000000,1284.000000,70.000000,70.000000,24.000000,56.000000,7.000000,19.000000,13.000000,19.000000,...,3.000000,18.000000,24.000000,56.000000,7.000000,19.000000,10.000000,23.000000,13.000000,0.500000
75%,2020.000000,1378.000000,78.000000,78.000000,28.000000,61.000000,9.000000,24.000000,18.000000,25.000000,...,5.000000,21.000000,28.000000,61.000000,9.000000,24.000000,13.000000,27.000000,16.000000,1.000000
max,2026.000000,1481.000000,149.000000,149.000000,57.000000,106.000000,26.000000,59.000000,50.000000,67.000000,...,21.000000,45.000000,57.000000,106.000000,26.000000,59.000000,38.000000,54.000000,41.000000,1.000000


In [99]:
#1.all_games

In [100]:
for i in all_games['Season']:
    print(i)

2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003
2003


2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016
2016


2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008
2008


2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026
2026


In [101]:
team_season_stats = all_games.groupby(['Season','TeamID']).agg(
    Games = ('Win','count'),
    Wins = ('Win','sum'),
    AvgScore    = ('Score', 'mean'),
    AvgOppScore = ('OppScore', 'mean'),
    AvgMargin   = ('Score', 'mean'),
    FGM   = ('FGM', 'mean'),   FGA  = ('FGA', 'mean'),
    FGM3  = ('FGM3', 'mean'),  FGA3 = ('FGA3', 'mean'),
    FTM   = ('FTM', 'mean'),   FTA  = ('FTA', 'mean'),
    OR_avg  = ('OR', 'mean'),  DR_avg = ('DR', 'mean'),
    Ast     = ('Ast', 'mean'), TO_avg = ('TO', 'mean'),
    Stl     = ('Stl', 'mean'), Blk    = ('Blk', 'mean'),
    PF_avg  = ('PF', 'mean'),
    Opp_FGM  = ('Opp_FGM', 'mean'),  Opp_FGA = ('Opp_FGA', 'mean'),
    Opp_FGM3 = ('Opp_FGM3', 'mean'), Opp_FGA3= ('Opp_FGA3', 'mean'),
    Opp_OR   = ('Opp_OR', 'mean'),   Opp_DR  = ('Opp_DR', 'mean'),
    Opp_TO   = ('Opp_TO', 'mean'),
).reset_index()

In [102]:
team_season_stats['AvgMargin'] = team_season_stats['AvgScore'] - team_season_stats['AvgOppScore']
team_season_stats['WinPct']    = team_season_stats['Wins'] / team_season_stats['Games']

In [103]:
team_season_stats.sample(5)

,Season,TeamID,Games,Wins,AvgScore,AvgOppScore,AvgMargin,FGM,FGA,FGM3,...,Blk,PF_avg,Opp_FGM,Opp_FGA,Opp_FGM3,Opp_FGA3,Opp_OR,Opp_DR,Opp_TO,WinPct
6714,2022,1288,23,9,70.173913,74.695652,-4.521739,25.652174,61.826087,6.043478,...,3.304348,20.260870,25.043478,55.782609,7.478261,21.565217,8.217391,24.086957,16.869565,0.391304
7657,2025,1146,31,7,68.483871,79.354839,-10.870968,23.903226,59.225806,8.387097,...,2.354839,13.903226,28.516129,61.032258,12.322581,31.806452,8.096774,24.419355,10.709677,0.225806
7011,2023,1227,31,11,66.677419,72.387097,-5.709677,23.096774,55.645161,8.000000,...,4.516129,17.516129,25.967742,57.322581,6.774194,19.774194,8.129032,26.064516,13.225806,0.354839
3873,2014,1255,28,4,66.892857,81.642857,-14.750000,24.214286,57.250000,5.535714,...,1.714286,19.678571,27.250000,56.285714,8.214286,21.785714,11.357143,26.107143,12.607143,0.142857
3447,2013,1176,30,21,65.433333,55.700000,9.733333,23.233333,48.300000,8.166667,...,4.433333,17.500000,19.300000,44.700000,3.600000,10.966667,9.833333,21.100000,15.866667,0.700000


In [104]:
#We can do possesion
#we have field goals attempted and made 

In [105]:
s = team_season_stats

s['Poss']     = s['FGA'] - s['OR_avg'] + s['TO_avg'] + 0.475 * s['FTA']
s['Opp_Poss'] = s['Opp_FGA'] - s['Opp_OR'] + s['Opp_TO'] + 0.475 * s['FTA']  # approx


s['eFG_pct']    = (s['FGM'] + 0.5 * s['FGM3']) / s['FGA']

s['TO_rate']    = s['TO_avg'] / s['Poss']

s['OR_rate']    = s['OR_avg'] / (s['OR_avg'] + s['Opp_DR'])

s['FT_rate']    = s['FTM'] / s['FGA']

s['Opp_eFG_pct'] = (s['Opp_FGM'] + 0.5 * s['Opp_FGM3']) / s['Opp_FGA']
s['Opp_TO_rate']  = s['Opp_TO'] / s['Opp_Poss']

s['Tempo'] = s['Poss']  # possessions per game ≈ tempo


s['Ast_TO_ratio'] = s['Ast'] / s['TO_avg']

team_season_stats = s

team_season_stats[['Season','TeamID','WinPct','eFG_pct','TO_rate','OR_rate','FT_rate','Tempo']].sample(5)

,Season,TeamID,WinPct,eFG_pct,TO_rate,OR_rate,FT_rate,Tempo
1452,2007,1250,0.344828,0.470668,0.230323,0.324468,0.287858,64.676724
4159,2015,1189,0.419355,0.474727,0.190265,0.278989,0.257323,69.004032
5108,2017,1441,0.233333,0.430653,0.233052,0.315331,0.182401,68.654167
2579,2010,1353,0.451613,0.480660,0.212174,0.334825,0.227531,68.416129
7056,2023,1272,0.764706,0.531649,0.177473,0.286965,0.267910,74.741912


In [106]:
team_season_stats.corr()

,Season,TeamID,Games,Wins,AvgScore,AvgOppScore,AvgMargin,FGM,FGA,FGM3,...,Poss,Opp_Poss,eFG_pct,TO_rate,OR_rate,FT_rate,Opp_eFG_pct,Opp_TO_rate,Tempo,Ast_TO_ratio
Season,1.000000,0.019589,-0.103692,-0.027472,0.245620,0.258969,0.001246,0.261515,0.297473,0.339680,...,0.235854,0.214435,0.161013,-0.639309,-0.510623,-0.196569,0.174511,-0.608352,0.235854,0.360218
TeamID,0.019589,1.000000,0.005749,0.058332,0.037030,-0.031342,0.058955,0.032403,0.013056,-0.014920,...,-0.009960,0.018156,0.023637,-0.060408,0.015929,0.039937,-0.043013,-0.038811,-0.009960,0.060476
Games,-0.103692,0.005749,1.000000,0.505062,0.074626,-0.260836,0.283856,0.074292,-0.021509,-0.023434,...,-0.153405,-0.058663,0.104353,-0.050591,0.175317,0.076243,-0.263841,0.058714,-0.153405,0.111458
Wins,-0.027472,0.058332,0.505062,1.000000,0.539729,-0.521652,0.913421,0.502729,0.116938,0.200163,...,-0.072063,0.163999,0.586964,-0.321857,0.320785,0.273804,-0.622783,0.053567,-0.072063,0.539981
AvgScore,0.245620,0.037030,0.074626,0.539729,1.000000,0.325361,0.613734,0.928479,0.672207,0.520018,...,0.627961,0.722769,0.688366,-0.510780,0.124025,0.237353,-0.063978,-0.230404,0.627961,0.606983
AvgOppScore,0.258969,-0.031342,-0.260836,-0.521652,0.325361,1.000000,-0.546870,0.285706,0.519167,0.247457,...,0.723951,0.554201,-0.063403,-0.095963,-0.270561,-0.058171,0.713679,-0.302442,0.723951,-0.100143
AvgMargin,0.001246,0.058955,0.283856,0.913421,0.613734,-0.546870,1.000000,0.583521,0.161693,0.253807,...,-0.048466,0.177208,0.662412,-0.372117,0.335714,0.258721,-0.652526,0.048524,-0.048466,0.621031
FGM,0.261515,0.032403,0.074292,0.502729,0.928479,0.285706,0.583521,1.000000,0.736101,0.421903,...,0.596070,0.632222,0.673001,-0.509920,0.116200,-0.055123,-0.069387,-0.208416,0.596070,0.634155
FGA,0.297473,0.013056,-0.021509,0.116938,0.672207,0.519167,0.161693,0.736101,1.000000,0.364592,...,0.777174,0.667394,0.062375,-0.472224,0.103315,-0.284811,0.102883,-0.136546,0.777174,0.342858
FGM3,0.339680,-0.014920,-0.023434,0.200163,0.520018,0.247457,0.253807,0.421903,0.364592,1.000000,...,0.253100,0.221871,0.562785,-0.471601,-0.339382,-0.228661,0.149431,-0.263075,0.253100,0.507430


In [107]:
seeds['SeedNum'] = seeds['Seed'].str[1:3].astype(int)

team_season_stats = team_season_stats.merge(
    seeds[['Season', 'TeamID', 'SeedNum']],
    on=['Season', 'TeamID'],
    how='left'   # LEFT JOIN: keep all teams, NaN if no seed
)



In [108]:
team_season_stats.shape

(8346, 39)

In [109]:
team_season_stats['SeedNum'].isnull().sum()

6874

In [110]:
len(team_season_stats['SeedNum'])

8346

In [111]:
team_season_stats['SeedNum']

0        NaN
1        NaN
2       10.0
3        NaN
4        NaN
        ... 
8341     NaN
8342     NaN
8343     NaN
8344     NaN
8345     NaN
Name: SeedNum, Length: 8346, dtype: float64

In [112]:
team_conf.sample(5)

,Season,TeamID,ConfAbbrev
12967,2024,1419,sun_belt
6122,2005,1171,ivy
3926,1998,1212,swac
8530,2012,1197,meac
12538,2023,1350,a_ten


In [113]:
team_conf.isnull().sum()

Season        0
TeamID        0
ConfAbbrev    0
dtype: int64

In [114]:
team_season_stats = team_season_stats.merge(
    team_conf[['Season', 'TeamID', 'ConfAbbrev']],
    on=['Season', 'TeamID'],
    how='left'
)

print(f'After conference join: {team_season_stats.shape}')
team_season_stats[['Season','TeamID','WinPct','SeedNum','ConfAbbrev']].sample(5)

After conference join: (8346, 40)


,Season,TeamID,WinPct,SeedNum,ConfAbbrev
6660,2022,1234,0.742857,5.0,big_ten
2866,2011,1288,0.548387,NaN,meac
8345,2026,1481,0.368421,NaN,nec
1253,2006,1398,0.444444,NaN,ovc
7090,2023,1308,0.285714,NaN,wac


In [115]:
team_season_stats['SeedNum'].isnull().sum()

6874

In [116]:
coach_main = coaches.sort_values('LastDayNum', ascending=False).drop_duplicates(
    subset=['Season', 'TeamID'], keep='first'
)[['Season', 'TeamID', 'CoachName']]

In [117]:
coach_exp = coaches.groupby('CoachName')['Season'].min().reset_index()
coach_exp.columns = ['CoachName', 'FirstSeason']

In [118]:
coach_main = coach_main.merge(coach_exp, on='CoachName', how='left')
coach_main['CoachYears'] = coach_main['Season'] - coach_main['FirstSeason']

In [119]:
team_season_stats = team_season_stats.merge(
    coach_main[['Season', 'TeamID', 'CoachName', 'CoachYears']],
    on=['Season', 'TeamID'],
    how='left'
)

In [120]:
team_season_stats[['Season','TeamID','WinPct','SeedNum','ConfAbbrev','CoachName','CoachYears']].sample(5)

,Season,TeamID,WinPct,SeedNum,ConfAbbrev,CoachName,CoachYears
2391,2010,1156,0.451613,NaN,horizon,gary_waters,13
632,2004,1442,0.111111,NaN,mid_cont,derek_thomas,0
6059,2020,1336,0.677419,NaN,big_ten,patrick_chambers,10
3204,2012,1282,0.333333,NaN,summit,matt_brown,4
1906,2008,1373,0.687500,13.0,maac,fran_mccaffery,22


In [121]:
for i, col in enumerate(team_season_stats.columns):
    print(f'  {i+1:2d}. {col}')

print(f'\nNull counts:')
nulls = team_season_stats.isnull().sum()
print(nulls[nulls > 0])

   1. Season
   2. TeamID
   3. Games
   4. Wins
   5. AvgScore
   6. AvgOppScore
   7. AvgMargin
   8. FGM
   9. FGA
  10. FGM3
  11. FGA3
  12. FTM
  13. FTA
  14. OR_avg
  15. DR_avg
  16. Ast
  17. TO_avg
  18. Stl
  19. Blk
  20. PF_avg
  21. Opp_FGM
  22. Opp_FGA
  23. Opp_FGM3
  24. Opp_FGA3
  25. Opp_OR
  26. Opp_DR
  27. Opp_TO
  28. WinPct
  29. Poss
  30. Opp_Poss
  31. eFG_pct
  32. TO_rate
  33. OR_rate
  34. FT_rate
  35. Opp_eFG_pct
  36. Opp_TO_rate
  37. Tempo
  38. Ast_TO_ratio
  39. SeedNum
  40. ConfAbbrev
  41. CoachName
  42. CoachYears

Null counts:
SeedNum    6874
dtype: int64


In [122]:
team_season_stats.isnull().sum()

Season             0
TeamID             0
Games              0
Wins               0
AvgScore           0
AvgOppScore        0
AvgMargin          0
FGM                0
FGA                0
FGM3               0
FGA3               0
FTM                0
FTA                0
OR_avg             0
DR_avg             0
Ast                0
TO_avg             0
Stl                0
Blk                0
PF_avg             0
Opp_FGM            0
Opp_FGA            0
Opp_FGM3           0
Opp_FGA3           0
Opp_OR             0
Opp_DR             0
Opp_TO             0
WinPct             0
Poss               0
Opp_Poss           0
eFG_pct            0
TO_rate            0
OR_rate            0
FT_rate            0
Opp_eFG_pct        0
Opp_TO_rate        0
Tempo              0
Ast_TO_ratio       0
SeedNum         6874
ConfAbbrev         0
CoachName          0
CoachYears         0
dtype: int64

In [123]:
team_season_stats[team_season_stats['TeamID'] == 1181].tail(3) 

,Season,TeamID,Games,Wins,AvgScore,AvgOppScore,AvgMargin,FGM,FGA,FGM3,...,OR_rate,FT_rate,Opp_eFG_pct,Opp_TO_rate,Tempo,Ast_TO_ratio,SeedNum,ConfAbbrev,CoachName,CoachYears
7329,2024,1181,32,24,79.843750,67.437500,12.406250,28.531250,59.218750,8.343750,...,0.295385,0.243799,0.490233,0.155180,69.093750,1.636667,4.0,acc,jon_scheyer,1
7691,2025,1181,34,31,82.705882,61.911765,20.794118,28.852941,59.117647,10.117647,...,0.340726,0.251741,0.445010,0.154211,67.393382,1.821086,1.0,acc,jon_scheyer,2
8055,2026,1181,22,21,84.136364,63.590909,20.545455,29.136364,58.363636,8.772727,...,0.360182,0.292835,0.458559,0.170678,69.835227,1.566667,NaN,acc,jon_scheyer,3


In [124]:
team_season_stats.nunique()

Season            24
TeamID           372
Games             28
Wins              35
AvgScore        4215
AvgOppScore     4123
AvgMargin       5465
FGM             2459
FGA             3264
FGM3            1855
FGA3            3328
FTM             2439
FTA             2981
OR_avg          2380
DR_avg          2301
Ast             2240
TO_avg          2250
Stl             1706
Blk             1488
PF_avg          2287
Opp_FGM         2458
Opp_FGA         3358
Opp_FGM3        1598
Opp_FGA3        3000
Opp_OR          2100
Opp_DR          2389
Opp_TO          2343
WinPct           336
Poss            8062
Opp_Poss        8080
eFG_pct         8192
TO_rate         8339
OR_rate         7886
FT_rate         8073
Opp_eFG_pct     8153
Opp_TO_rate     8339
Tempo           8062
Ast_TO_ratio    7637
SeedNum           16
ConfAbbrev        36
CoachName       1102
CoachYears        41
dtype: int64

In [125]:
games_w = tourney_compact[['Season','WTeamID','LTeamID']].copy()
games_w.columns = ['Season', 'TeamA', 'TeamB']
games_w['Result'] = 1   # TeamA won

games_l = tourney_compact[['Season','WTeamID','LTeamID']].copy()
games_l.columns = ['Season', 'TeamB', 'TeamA']   # SWAP!
games_l['Result'] = 0   # TeamA lost

tourney_games = pd.concat([games_w, games_l], ignore_index=True)
print(f'Tournament game rows: {len(tourney_games)}')
tourney_games.head()

Tournament game rows: 5170


,Season,TeamA,TeamB,Result
0,1985,1116,1234,1
1,1985,1120,1345,1
2,1985,1207,1250,1
3,1985,1229,1425,1
4,1985,1242,1325,1


In [126]:
len(team_season_stats['Season'] == 1985)

8346

In [127]:
tourney_games = tourney_games.merge(
    team_season_stats,
    left_on=['Season', 'TeamA'],
    right_on=['Season', 'TeamID'],
    how='left'
)

In [128]:
tourney_games = tourney_games.merge(
    team_season_stats,
    left_on=['Season', 'TeamB'],
    right_on=['Season', 'TeamID'],
    how='left',
    suffixes=('_A', '_B')
)

In [129]:
tourney_games.sample(5)

,Season,TeamA,TeamB,Result,TeamID_A,Games_A,Wins_A,AvgScore_A,AvgOppScore_A,AvgMargin_A,...,OR_rate_B,FT_rate_B,Opp_eFG_pct_B,Opp_TO_rate_B,Tempo_B,Ast_TO_ratio_B,SeedNum_B,ConfAbbrev_B,CoachName_B,CoachYears_B
692,1995,1417,1116,1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5157,2025,1268,1196,0,1268.0,33.0,25.0,81.666667,67.000000,14.666667,...,0.370599,0.233808,0.453488,0.157952,71.816912,1.544413,1.0,sec,todd_golden,5.0
2036,2016,1328,1401,1,1328.0,32.0,25.0,80.406250,70.437500,9.968750,...,0.350482,0.251984,0.470725,0.202701,68.812500,1.431421,3.0,sec,billy_kennedy,18.0
2082,2017,1376,1266,1,1376.0,31.0,21.0,71.483871,64.548387,6.935484,...,0.272637,0.242507,0.519412,0.188027,71.297581,1.357143,10.0,big_east,steve_wojciechowski,2.0
1234,2004,1163,1177,1,1163.0,33.0,27.0,79.090909,63.939394,15.151515,...,0.380252,0.265730,0.489917,0.184748,65.718333,1.022333,7.0,cusa,dave_leitao,9.0


In [130]:
len(tourney_games)

5170

In [131]:
tourney_games.shape

(5170, 86)

In [132]:
diff_features = [
    'WinPct', 'AvgMargin', 'AvgScore', 'AvgOppScore',
    'eFG_pct', 'TO_rate', 'OR_rate', 'FT_rate',
    'Opp_eFG_pct', 'Opp_TO_rate',
    'Tempo', 'Ast_TO_ratio',
    'SeedNum',
    'CoachYears'
]

In [133]:
for f in diff_features:
    tourney_games[f'Diff_{f}'] = tourney_games[f'{f}_A'] - tourney_games[f'{f}_B']

In [134]:
feature_cols = [f'Diff_{f}' for f in diff_features]
print(tourney_games[feature_cols].isnull().sum())

Diff_WinPct          2272
Diff_AvgMargin       2272
Diff_AvgScore        2272
Diff_AvgOppScore     2272
Diff_eFG_pct         2272
Diff_TO_rate         2272
Diff_OR_rate         2272
Diff_FT_rate         2272
Diff_Opp_eFG_pct     2272
Diff_Opp_TO_rate     2272
Diff_Tempo           2272
Diff_Ast_TO_ratio    2272
Diff_SeedNum         2272
Diff_CoachYears      2272
dtype: int64


In [135]:
df_model = tourney_games[['Season','TeamA','TeamB','Result'] + feature_cols].dropna()

X = df_model[feature_cols]
y = df_model['Result']

print(f'Final dataset: {X.shape[0]} rows × {X.shape[1]} features')
print(f'Target balance: \n{y.value_counts()}')

Final dataset: 2898 rows × 14 features
Target balance: 
Result
1    1449
0    1449
Name: count, dtype: int64


In [136]:
corr = X.corrwith(y).sort_values(ascending=False)
print(corr)

Diff_AvgMargin       0.404873
Diff_WinPct          0.325587
Diff_Ast_TO_ratio    0.248584
Diff_AvgScore        0.231600
Diff_OR_rate         0.214340
Diff_CoachYears      0.208571
Diff_eFG_pct         0.179187
Diff_Tempo           0.022024
Diff_Opp_TO_rate     0.011476
Diff_FT_rate        -0.053923
Diff_AvgOppScore    -0.148562
Diff_TO_rate        -0.187046
Diff_Opp_eFG_pct    -0.219309
Diff_SeedNum        -0.482312
dtype: float64


In [137]:
from sklearn.feature_selection import mutual_info_classif

mi = pd.Series(
    mutual_info_classif(X, y, random_state=42),
    index=X.columns
).sort_values(ascending=False)

print(mi)

Diff_SeedNum         0.132933
Diff_AvgMargin       0.097801
Diff_WinPct          0.077472
Diff_AvgScore        0.060402
Diff_OR_rate         0.042121
Diff_TO_rate         0.032950
Diff_eFG_pct         0.031261
Diff_Ast_TO_ratio    0.028799
Diff_AvgOppScore     0.014077
Diff_Tempo           0.006156
Diff_CoachYears      0.006054
Diff_Opp_eFG_pct     0.001890
Diff_FT_rate         0.000000
Diff_Opp_TO_rate     0.000000
dtype: float64


In [138]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

corr.plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Correlation with Result')

mi.plot.barh(ax=axes[1], color='coral')
axes[1].set_title('Mutual Information')

coefs.plot.barh(ax=axes[2], color='seagreen')
axes[2].set_title('LogReg Coefficients (standardized)')

plt.tight_layout()
plt.show()

In [139]:
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

In [140]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_scaled, y)

coefs = pd.Series(
    lr_model.coef_[0],
    index=X.columns
).sort_values(key=abs, ascending=False)

print("Standardized LogReg Coefficients (|largest| = most important):")
print(coefs)

Standardized LogReg Coefficients (|largest| = most important):
Diff_SeedNum        -0.956031
Diff_Tempo          -0.531008
Diff_AvgMargin       0.528526
Diff_AvgScore        0.514881
Diff_Ast_TO_ratio   -0.248279
Diff_WinPct         -0.221531
Diff_FT_rate        -0.162735
Diff_Opp_TO_rate     0.117369
Diff_TO_rate        -0.107312
Diff_eFG_pct        -0.094702
Diff_CoachYears      0.055742
Diff_OR_rate         0.051570
Diff_AvgOppScore     0.048374
Diff_Opp_eFG_pct     0.009421
dtype: float64


In [141]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

corr.plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Correlation with Result')
axes[0].axvline(x=0, color='black', linewidth=0.5)

mi.plot.barh(ax=axes[1], color='coral')
axes[1].set_title('Mutual Information')

coefs.sort_values().plot.barh(ax=axes[2], color='seagreen')
axes[2].set_title('LogReg Coefficients (standardized)')
axes[2].axvline(x=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig("output/feature_importance_comparison.png", dpi=150)
plt.show()

In [142]:
SPLIT_YEAR = 2019

train_mask = df_model['Season'] <= SPLIT_YEAR
test_mask  = df_model['Season'] > SPLIT_YEAR

X_train_raw = X[train_mask]
X_test_raw  = X[test_mask]
y_train     = y[train_mask]
y_test      = y[test_mask]

# Scale using ONLY training data (prevents data leakage)
scaler = StandardScaler()
X_train = pd.DataFrame(
    scaler.fit_transform(X_train_raw),      # fit on train
    columns=X.columns, index=X_train_raw.index
)
X_test = pd.DataFrame(
    scaler.transform(X_test_raw),           # only transform test
    columns=X.columns, index=X_test_raw.index
)

In [143]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import (
    log_loss, accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
)
from sklearn.inspection import permutation_importance

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(
                               n_estimators=200,
                               max_depth=6,         # keep small to avoid overfit
                               min_samples_leaf=10,  # same reason
                               random_state=42
                           ),
    'Gradient Boosting':   GradientBoostingClassifier(
                               n_estimators=200,
                               max_depth=3,          # shallow trees = less overfit
                               learning_rate=0.05,   # slow learning = better generalization
                               min_samples_leaf=10,
                               random_state=42
                           ),
}

In [144]:
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)

    # Predict probabilities (needed for log_loss)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

    ll  = log_loss(y_test, y_prob)
    acc = accuracy_score(y_test, y_pred)

    results[name] = {
        'model': model,
        'log_loss': ll,
        'accuracy': acc,
        'y_prob': y_prob,
        'y_pred': y_pred
    }


In [145]:
print(f"\n\n{'Model':<25} {'Log Loss':>10} {'Accuracy':>10}")
for name, r in results.items():
    print(f"{name:<25} {r['log_loss']:>10.4f} {r['accuracy']:>10.4f}")



Model                       Log Loss   Accuracy
Logistic Regression           0.5989     0.6796
Random Forest                 0.5964     0.6796
Gradient Boosting             0.6210     0.6751


In [146]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=['Loss', 'Win']).plot(ax=ax, cmap='Blues')
    ax.set_title(f"{name}\nAcc={r['accuracy']:.3f}")

plt.tight_layout()
plt.savefig("output/confusion_matrices.png", dpi=150)
plt.show()


In [147]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['steelblue', 'coral', 'seagreen']

for (name, r), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig("output/roc_curves.png", dpi=150)
plt.show()

In [148]:
best_name = min(results, key=lambda k: results[k]['log_loss'])
best_model = results[best_name]['model']
print(f"Best model: {best_name} (Log Loss: {results[best_name]['log_loss']:.4f})")

perm = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=30, random_state=42, scoring='neg_log_loss'
)

perm_imp = pd.Series(
    perm.importances_mean,
    index=X.columns
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
perm_imp.sort_values().plot.barh(ax=ax, color='teal')
ax.set_title(f'Permutation Importance ({best_name})')
ax.set_xlabel('Mean decrease in neg_log_loss when feature is shuffled')
plt.tight_layout()
plt.savefig("output/permutation_importance.png", dpi=150)
plt.show()

print("\nPermutation Importance Rankings:")
print(perm_imp)

Best model: Random Forest (Log Loss: 0.5964)

Permutation Importance Rankings:
Diff_SeedNum         0.058012
Diff_AvgMargin       0.009143
Diff_OR_rate         0.003983
Diff_Ast_TO_ratio    0.003579
Diff_WinPct          0.003474
Diff_Opp_eFG_pct     0.002115
Diff_AvgScore        0.001835
Diff_eFG_pct         0.001498
Diff_TO_rate         0.001354
Diff_CoachYears      0.000334
Diff_Opp_TO_rate    -0.000089
Diff_Tempo          -0.000645
Diff_FT_rate        -0.000800
Diff_AvgOppScore    -0.002851
dtype: float64


In [149]:
df_sorted = df_model.sort_values('Season')
X_sorted = df_sorted[feature_cols]
y_sorted = df_sorted['Result']

X_sorted_scaled = pd.DataFrame(
    StandardScaler().fit_transform(X_sorted),
    columns=X_sorted.columns,
    index=X_sorted.index
)

tscv = TimeSeriesSplit(n_splits=5)

print(f"\nTime-Series Cross-Validation (5 splits):")
print(f"{'Model':<25} {'Mean Log Loss':>14} {'Std':>8}")
print("-" * 49)

for name, model_template in models.items():
    # Use fresh model for CV
    if name == 'Logistic Regression':
        m = LogisticRegression(max_iter=1000, random_state=42)
    elif name == 'Random Forest':
        m = RandomForestClassifier(n_estimators=200, max_depth=6,
                                   min_samples_leaf=10, random_state=42)
    else:
        m = GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                       learning_rate=0.05, min_samples_leaf=10,
                                       random_state=42)

    scores = cross_val_score(m, X_sorted_scaled, y_sorted,
                             cv=tscv, scoring='neg_log_loss')
    print(f"{name:<25} {-scores.mean():>14.4f} {scores.std():>8.4f}")



Time-Series Cross-Validation (5 splits):
Model                      Mean Log Loss      Std
-------------------------------------------------
Logistic Regression               0.5690   0.0430
Random Forest                     0.5801   0.0350
Gradient Boosting                 0.6015   0.0368


In [150]:
print(classification_report(y_test, results[best_name]['y_pred'],
                            target_names=['TeamA Lost', 'TeamA Won']))

              precision    recall  f1-score   support

  TeamA Lost       0.68      0.69      0.68       334
   TeamA Won       0.68      0.67      0.68       334

    accuracy                           0.68       668
   macro avg       0.68      0.68      0.68       668
weighted avg       0.68      0.68      0.68       668



In [151]:
final_scaler = StandardScaler()
X_all_scaled = final_scaler.fit_transform(X)

if best_name == 'Logistic Regression':
    final_model = LogisticRegression(max_iter=1000, random_state=42)
elif best_name == 'Random Forest':
    final_model = RandomForestClassifier(n_estimators=200, max_depth=6,
                                         min_samples_leaf=10, random_state=42)
else:
    final_model = GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                              learning_rate=0.05,
                                              min_samples_leaf=10, random_state=42)

final_model.fit(X_all_scaled, y)

joblib.dump(final_model, 'output/best_model.pkl')
joblib.dump(final_scaler, 'output/scaler.pkl')


['output/scaler.pkl']

# ============================================================
# PART 2: FULL SUBMISSION PIPELINE (Men + Women)
# ============================================================
# Steps:
# 1. Gender-parameterized data loading & feature engineering
# 2. Massey Ordinals (men only)
# 3. SeedNum imputation for non-tournament teams
# 4. SOS + Conference Strength
# 5. Train separate models per gender
# 6. Generate Stage 1 + Stage 2 submission CSVs

In [152]:
# ── Step 1: Gender-parameterized feature engineering ──────────────────────

def build_team_season_stats(prefix, data_dir):
    """
    Build team_season_stats for either Men ('M') or Women ('W').
    Returns a DataFrame with one row per (Season, TeamID) with all features.
    """
    # Load detailed regular season results
    reg_detail = pd.read_csv(data_dir / f"{prefix}RegularSeasonDetailedResults.csv")
    
    # Load seeds
    seed_df = pd.read_csv(data_dir / f"{prefix}NCAATourneySeeds.csv")
    seed_df['SeedNum'] = seed_df['Seed'].str[1:3].astype(int)
    
    # Load conferences
    conf_df = pd.read_csv(data_dir / f"{prefix}TeamConferences.csv")
    
    # ── Unpivot winners and losers ──
    w_cols = {
        'Season': 'Season', 'WTeamID': 'TeamID',
        'WScore': 'Score', 'LScore': 'OppScore',
        'WFGM': 'FGM', 'WFGA': 'FGA',
        'WFGM3': 'FGM3', 'WFGA3': 'FGA3',
        'WFTM': 'FTM', 'WFTA': 'FTA',
        'WOR': 'OR', 'WDR': 'DR',
        'WAst': 'Ast', 'WTO': 'TO',
        'WStl': 'Stl', 'WBlk': 'Blk', 'WPF': 'PF',
        'LFGM': 'Opp_FGM', 'LFGA': 'Opp_FGA',
        'LFGM3': 'Opp_FGM3', 'LFGA3': 'Opp_FGA3',
        'LOR': 'Opp_OR', 'LDR': 'Opp_DR',
        'LTO': 'Opp_TO'
    }
    winners = reg_detail[list(w_cols.keys())].rename(columns=w_cols)
    winners['Win'] = 1

    l_cols = {
        'Season': 'Season', 'LTeamID': 'TeamID',
        'LScore': 'Score', 'WScore': 'OppScore',
        'LFGM': 'FGM', 'LFGA': 'FGA',
        'LFGM3': 'FGM3', 'LFGA3': 'FGA3',
        'LFTM': 'FTM', 'LFTA': 'FTA',
        'LOR': 'OR', 'LDR': 'DR',
        'LAst': 'Ast', 'LTO': 'TO',
        'LStl': 'Stl', 'LBlk': 'Blk', 'LPF': 'PF',
        'WFGM': 'Opp_FGM', 'WFGA': 'Opp_FGA',
        'WFGM3': 'Opp_FGM3', 'WFGA3': 'Opp_FGA3',
        'WOR': 'Opp_OR', 'WDR': 'Opp_DR',
        'WTO': 'Opp_TO'
    }
    losers = reg_detail[list(l_cols.keys())].rename(columns=l_cols)
    losers['Win'] = 0

    all_games = pd.concat([winners, losers], ignore_index=True)

    # ── Aggregate to team-season level ──
    stats = all_games.groupby(['Season', 'TeamID']).agg(
        Games       = ('Win', 'count'),
        Wins        = ('Win', 'sum'),
        AvgScore    = ('Score', 'mean'),
        AvgOppScore = ('OppScore', 'mean'),
        FGM   = ('FGM', 'mean'),   FGA  = ('FGA', 'mean'),
        FGM3  = ('FGM3', 'mean'),  FGA3 = ('FGA3', 'mean'),
        FTM   = ('FTM', 'mean'),   FTA  = ('FTA', 'mean'),
        OR_avg  = ('OR', 'mean'),  DR_avg = ('DR', 'mean'),
        Ast     = ('Ast', 'mean'), TO_avg = ('TO', 'mean'),
        Stl     = ('Stl', 'mean'), Blk    = ('Blk', 'mean'),
        PF_avg  = ('PF', 'mean'),
        Opp_FGM  = ('Opp_FGM', 'mean'),  Opp_FGA = ('Opp_FGA', 'mean'),
        Opp_FGM3 = ('Opp_FGM3', 'mean'), Opp_FGA3= ('Opp_FGA3', 'mean'),
        Opp_OR   = ('Opp_OR', 'mean'),   Opp_DR  = ('Opp_DR', 'mean'),
        Opp_TO   = ('Opp_TO', 'mean'),
    ).reset_index()

    # ── Derived features ──
    stats['AvgMargin']   = stats['AvgScore'] - stats['AvgOppScore']
    stats['WinPct']      = stats['Wins'] / stats['Games']
    stats['Poss']        = stats['FGA'] - stats['OR_avg'] + stats['TO_avg'] + 0.475 * stats['FTA']
    stats['Opp_Poss']    = stats['Opp_FGA'] - stats['Opp_OR'] + stats['Opp_TO'] + 0.475 * stats['FTA']
    stats['eFG_pct']     = (stats['FGM'] + 0.5 * stats['FGM3']) / stats['FGA']
    stats['TO_rate']     = stats['TO_avg'] / stats['Poss']
    stats['OR_rate']     = stats['OR_avg'] / (stats['OR_avg'] + stats['Opp_DR'])
    stats['FT_rate']     = stats['FTM'] / stats['FGA']
    stats['Opp_eFG_pct'] = (stats['Opp_FGM'] + 0.5 * stats['Opp_FGM3']) / stats['Opp_FGA']
    stats['Opp_TO_rate'] = stats['Opp_TO'] / stats['Opp_Poss']
    stats['Tempo']       = stats['Poss']
    stats['Ast_TO_ratio']= stats['Ast'] / stats['TO_avg']

    # ── Merge seeds ──
    stats = stats.merge(
        seed_df[['Season', 'TeamID', 'SeedNum']],
        on=['Season', 'TeamID'], how='left'
    )

    # ── Merge conferences ──
    stats = stats.merge(
        conf_df[['Season', 'TeamID', 'ConfAbbrev']],
        on=['Season', 'TeamID'], how='left'
    )

    # ── Coaching data (men only) ──
    if prefix == 'M':
        coaches_df = pd.read_csv(data_dir / "MTeamCoaches.csv")
        coach_main = coaches_df.sort_values('LastDayNum', ascending=False).drop_duplicates(
            subset=['Season', 'TeamID'], keep='first'
        )[['Season', 'TeamID', 'CoachName']]
        coach_exp = coaches_df.groupby('CoachName')['Season'].min().reset_index()
        coach_exp.columns = ['CoachName', 'FirstSeason']
        coach_main = coach_main.merge(coach_exp, on='CoachName', how='left')
        coach_main['CoachYears'] = coach_main['Season'] - coach_main['FirstSeason']
        stats = stats.merge(
            coach_main[['Season', 'TeamID', 'CoachYears']],
            on=['Season', 'TeamID'], how='left'
        )
    else:
        stats['CoachYears'] = 0  # placeholder for women (no data available)

    stats['Gender'] = prefix
    
    # Return all_games too (needed for SOS computation)
    return stats, all_games

# Build for both genders
print("Building Men's team_season_stats...")
m_stats, m_all_games = build_team_season_stats('M', DATA_DIR)
print(f"  Men: {m_stats.shape[0]} team-seasons, seasons {m_stats['Season'].min()}-{m_stats['Season'].max()}")

print("Building Women's team_season_stats...")
w_stats, w_all_games = build_team_season_stats('W', DATA_DIR)
print(f"  Women: {w_stats.shape[0]} team-seasons, seasons {w_stats['Season'].min()}-{w_stats['Season'].max()}")

# Combine
all_stats = pd.concat([m_stats, w_stats], ignore_index=True)
print(f"\nCombined: {all_stats.shape[0]} team-seasons")
print(f"Men seeds null: {m_stats['SeedNum'].isnull().sum()}/{len(m_stats)}")
print(f"Women seeds null: {w_stats['SeedNum'].isnull().sum()}/{len(w_stats)}")

Building Men's team_season_stats...
  Men: 8346 team-seasons, seasons 2003-2026
Building Women's team_season_stats...
  Women: 5965 team-seasons, seasons 2010-2026

Combined: 14311 team-seasons
Men seeds null: 6874/8346
Women seeds null: 4989/5965


In [155]:
# ── Step 3: Massey Ordinals (men only) ───────────────────────────────────
print("Loading Massey Ordinals (this may take a moment)...")
massey = pd.read_csv(DATA_DIR / "MMasseyOrdinals.csv")
print(f"  Loaded {len(massey):,} rows, {massey['SystemName'].nunique()} ranking systems")

# For each season, get end-of-regular-season rankings (day <= 133)
# For 2026, the season is still in progress so use whatever is latest
massey_latest_day = massey.groupby(['Season', 'TeamID'])['RankingDayNum'].max().reset_index()
massey_latest_day.columns = ['Season', 'TeamID', 'LatestDay']

massey = massey.merge(massey_latest_day, on=['Season', 'TeamID'])
massey_end = massey[massey['RankingDayNum'] == massey['LatestDay']]

# Average rank across all systems (lower = better)
massey_avg = massey_end.groupby(['Season', 'TeamID'])['OrdinalRank'].mean().reset_index()
massey_avg.columns = ['Season', 'TeamID', 'MasseyAvgRank']

print(f"  Massey features computed for {len(massey_avg):,} team-seasons")
print(f"  2026 teams with Massey: {len(massey_avg[massey_avg['Season']==2026])}")

# Merge into men's stats
m_stats = m_stats.merge(massey_avg, on=['Season', 'TeamID'], how='left')
print(f"  Men's MasseyAvgRank null: {m_stats['MasseyAvgRank'].isnull().sum()}/{len(m_stats)}")

# Women don't have Massey data
w_stats['MasseyAvgRank'] = np.nan

# Recombine
all_stats = pd.concat([m_stats, w_stats], ignore_index=True)
print(f"\nall_stats shape: {all_stats.shape}")
all_stats[['Season','TeamID','Gender','WinPct','SeedNum','MasseyAvgRank']].sample(8)

Loading Massey Ordinals (this may take a moment)...
  Loaded 5,761,702 rows, 196 ranking systems
  Massey features computed for 8,356 team-seasons
  2026 teams with Massey: 365
  Men's MasseyAvgRank null: 0/8346

all_stats shape: (14311, 45)


,Season,TeamID,Gender,WinPct,SeedNum,MasseyAvgRank
6305,2021,1232,M,0.583333,NaN,123.576923
7429,2024,1283,M,0.515152,NaN,146.473684
5848,2020,1116,M,0.625000,NaN,50.596774
6758,2022,1335,M,0.428571,NaN,211.206349
10326,2015,3370,W,0.322581,NaN,NaN
11071,2017,3417,W,0.741935,4.0,NaN
150,2003,1268,M,0.678571,6.0,25.882353
801,2005,1265,M,0.392857,NaN,256.882353


In [156]:
# ── Step 4: Impute SeedNum for non-tournament teams ──────────────────────
from sklearn.linear_model import Ridge

def impute_seeds(stats_df, use_massey=False):
    """Impute SeedNum for teams without tournament seeds using Ridge regression."""
    impute_features = ['WinPct', 'AvgMargin', 'eFG_pct', 'TO_rate', 'OR_rate', 
                        'FT_rate', 'Opp_eFG_pct', 'Tempo']
    if use_massey:
        impute_features.append('MasseyAvgRank')
    
    has_seed = stats_df['SeedNum'].notna()
    
    # Only use rows where all imputation features are available
    train_mask = has_seed & stats_df[impute_features].notna().all(axis=1)
    predict_mask = ~has_seed & stats_df[impute_features].notna().all(axis=1)
    
    if train_mask.sum() == 0 or predict_mask.sum() == 0:
        return stats_df
    
    ridge = Ridge(alpha=1.0)
    ridge.fit(stats_df.loc[train_mask, impute_features], stats_df.loc[train_mask, 'SeedNum'])
    
    predicted_seeds = ridge.predict(stats_df.loc[predict_mask, impute_features])
    stats_df.loc[predict_mask, 'SeedNum'] = np.clip(predicted_seeds, 1, 16)
    
    return stats_df

# Impute for men (with Massey) and women (without)
m_before_null = m_stats['SeedNum'].isnull().sum()
m_stats = impute_seeds(m_stats, use_massey=True)
print(f"Men SeedNum: {m_before_null} null -> {m_stats['SeedNum'].isnull().sum()} null")

w_before_null = w_stats['SeedNum'].isnull().sum()
w_stats = impute_seeds(w_stats, use_massey=False)
print(f"Women SeedNum: {w_before_null} null -> {w_stats['SeedNum'].isnull().sum()} null")

Men SeedNum: 6874 null -> 0 null
Women SeedNum: 4989 null -> 0 null


In [157]:
# ── Step 5: Strength of Schedule (SOS) + Conference Strength ─────────────

def compute_sos(all_games_df, stats_df):
    """Compute average opponent WinPct for each team-season."""
    # Get opponent IDs from the detailed results
    # all_games_df has: Season, TeamID, OppScore, Score, Win, etc.
    # We need to figure out opponent IDs from the original detailed results
    pass

# Simpler approach: compute SOS from regular season compact results
def compute_sos_from_compact(prefix, data_dir, stats_df):
    """Compute SOS from compact results which have both team IDs."""
    compact = pd.read_csv(data_dir / f"{prefix}RegularSeasonCompactResults.csv")
    
    # Build game list with both perspectives
    games_w = compact[['Season', 'WTeamID', 'LTeamID']].rename(
        columns={'WTeamID': 'TeamID', 'LTeamID': 'OppID'})
    games_l = compact[['Season', 'LTeamID', 'WTeamID']].rename(
        columns={'LTeamID': 'TeamID', 'WTeamID': 'OppID'})
    all_matchups = pd.concat([games_w, games_l], ignore_index=True)
    
    # Merge opponent WinPct
    opp_winpct = stats_df[['Season', 'TeamID', 'WinPct']].rename(
        columns={'TeamID': 'OppID', 'WinPct': 'OppWinPct'})
    all_matchups = all_matchups.merge(opp_winpct, on=['Season', 'OppID'], how='left')
    
    # Average opponent WinPct = SOS
    sos = all_matchups.groupby(['Season', 'TeamID'])['OppWinPct'].mean().reset_index()
    sos.columns = ['Season', 'TeamID', 'SOS']
    
    return sos

# Compute SOS for both genders
m_sos = compute_sos_from_compact('M', DATA_DIR, m_stats)
m_stats = m_stats.merge(m_sos, on=['Season', 'TeamID'], how='left')

w_sos = compute_sos_from_compact('W', DATA_DIR, w_stats)
w_stats = w_stats.merge(w_sos, on=['Season', 'TeamID'], how='left')

# Conference Strength: average WinPct of all teams in the conference that season
def compute_conf_strength(stats_df):
    conf_str = stats_df.groupby(['Season', 'ConfAbbrev'])['WinPct'].mean().reset_index()
    conf_str.columns = ['Season', 'ConfAbbrev', 'ConfStrength']
    return stats_df.merge(conf_str, on=['Season', 'ConfAbbrev'], how='left')

m_stats = compute_conf_strength(m_stats)
w_stats = compute_conf_strength(w_stats)

# Fill any remaining NaN in MasseyAvgRank for men with median
m_stats['MasseyAvgRank'] = m_stats['MasseyAvgRank'].fillna(m_stats['MasseyAvgRank'].median())

# Recombine final stats
all_stats = pd.concat([m_stats, w_stats], ignore_index=True)
print(f"Final all_stats: {all_stats.shape}")
print(f"\nNull counts:")
print(all_stats[['SeedNum','SOS','ConfStrength','MasseyAvgRank','CoachYears']].isnull().sum())

Final all_stats: (14311, 47)

Null counts:
SeedNum             0
SOS                 0
ConfStrength        0
MasseyAvgRank    5965
CoachYears          0
dtype: int64


In [158]:
# ── Step 6: Build tournament training data per gender ────────────────────

# Define feature sets per gender
MEN_FEATURES = [
    'WinPct', 'AvgMargin', 'AvgScore', 'AvgOppScore',
    'eFG_pct', 'TO_rate', 'OR_rate', 'FT_rate',
    'Opp_eFG_pct', 'Opp_TO_rate', 'Tempo', 'Ast_TO_ratio',
    'SeedNum', 'MasseyAvgRank', 'CoachYears', 'SOS', 'ConfStrength'
]

WOMEN_FEATURES = [
    'WinPct', 'AvgMargin', 'AvgScore', 'AvgOppScore',
    'eFG_pct', 'TO_rate', 'OR_rate', 'FT_rate',
    'Opp_eFG_pct', 'Opp_TO_rate', 'Tempo', 'Ast_TO_ratio',
    'SeedNum', 'SOS', 'ConfStrength'
]

def build_tourney_training(prefix, data_dir, stats_df, feature_list):
    """Build symmetric tournament training data with difference features."""
    tourney = pd.read_csv(data_dir / f"{prefix}NCAATourneyCompactResults.csv")
    
    # Symmetric: TeamA won + TeamA lost
    gw = tourney[['Season', 'WTeamID', 'LTeamID']].copy()
    gw.columns = ['Season', 'TeamA', 'TeamB']
    gw['Result'] = 1
    
    gl = tourney[['Season', 'WTeamID', 'LTeamID']].copy()
    gl.columns = ['Season', 'TeamB', 'TeamA']
    gl['Result'] = 0
    
    games = pd.concat([gw, gl], ignore_index=True)
    
    # Merge stats for both teams
    games = games.merge(stats_df, left_on=['Season', 'TeamA'], 
                        right_on=['Season', 'TeamID'], how='left')
    games = games.merge(stats_df, left_on=['Season', 'TeamB'], 
                        right_on=['Season', 'TeamID'], how='left', suffixes=('_A', '_B'))
    
    # Compute difference features
    diff_cols = []
    for f in feature_list:
        col_name = f'Diff_{f}'
        games[col_name] = games[f'{f}_A'] - games[f'{f}_B']
        diff_cols.append(col_name)
    
    # Drop rows with NaN in diff features
    df_model = games[['Season', 'TeamA', 'TeamB', 'Result'] + diff_cols].dropna()
    
    return df_model, diff_cols

# Build for men
m_tourney, m_diff_cols = build_tourney_training('M', DATA_DIR, m_stats, MEN_FEATURES)
print(f"Men tourney training: {len(m_tourney)} rows, {len(m_diff_cols)} features")
print(f"  Seasons: {m_tourney['Season'].min()}-{m_tourney['Season'].max()}")

# Build for women
w_tourney, w_diff_cols = build_tourney_training('W', DATA_DIR, w_stats, WOMEN_FEATURES)
print(f"Women tourney training: {len(w_tourney)} rows, {len(w_diff_cols)} features")
print(f"  Seasons: {w_tourney['Season'].min()}-{w_tourney['Season'].max()}")

Men tourney training: 2898 rows, 17 features
  Seasons: 2003-2025
Women tourney training: 1922 rows, 15 features
  Seasons: 2010-2025


In [159]:
# ── Step 7: Train & evaluate models per gender ──────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import log_loss, accuracy_score
import joblib

def train_and_evaluate(tourney_df, diff_cols, gender_label):
    """Train models, evaluate, and return best model + scaler."""
    X = tourney_df[diff_cols]
    y = tourney_df['Result']
    
    # Temporal split
    SPLIT_YEAR = 2019
    train_mask = tourney_df['Season'] <= SPLIT_YEAR
    test_mask  = tourney_df['Season'] > SPLIT_YEAR
    
    X_train_raw = X[train_mask]
    X_test_raw  = X[test_mask]
    y_train = y[train_mask]
    y_test  = y[test_mask]
    
    # Scale
    scaler = StandardScaler()
    X_train = pd.DataFrame(scaler.fit_transform(X_train_raw), columns=diff_cols, index=X_train_raw.index)
    X_test  = pd.DataFrame(scaler.transform(X_test_raw), columns=diff_cols, index=X_test_raw.index)
    
    models = {
        'LogReg': LogisticRegression(max_iter=1000, random_state=42),
        'RF':     RandomForestClassifier(n_estimators=300, max_depth=6, 
                                          min_samples_leaf=8, random_state=42),
        'GBM':    GradientBoostingClassifier(n_estimators=300, max_depth=3, 
                                              learning_rate=0.03, subsample=0.8,
                                              min_samples_leaf=8, random_state=42),
    }
    
    print(f"\n{'='*50}")
    print(f"  {gender_label} — Train: {train_mask.sum()}, Test: {test_mask.sum()}")
    print(f"{'='*50}")
    print(f"  {'Model':<10} {'Log Loss':>10} {'Accuracy':>10}")
    
    results = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_prob = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)
        ll  = log_loss(y_test, y_prob)
        acc = accuracy_score(y_test, y_pred)
        results[name] = {'model': model, 'log_loss': ll, 'accuracy': acc}
        print(f"  {name:<10} {ll:>10.4f} {acc:>10.4f}")
    
    # Select best by log loss
    best_name = min(results, key=lambda k: results[k]['log_loss'])
    print(f"\n  Best: {best_name} (log_loss={results[best_name]['log_loss']:.4f})")
    
    # Time-series CV on full data
    X_sorted = tourney_df.sort_values('Season')[diff_cols]
    y_sorted = tourney_df.sort_values('Season')['Result']
    full_scaler = StandardScaler()
    X_sorted_scaled = pd.DataFrame(full_scaler.fit_transform(X_sorted), 
                                    columns=diff_cols, index=X_sorted.index)
    
    tscv = TimeSeriesSplit(n_splits=5)
    for name in models:
        if name == 'LogReg':
            m = LogisticRegression(max_iter=1000, random_state=42)
        elif name == 'RF':
            m = RandomForestClassifier(n_estimators=300, max_depth=6, 
                                        min_samples_leaf=8, random_state=42)
        else:
            m = GradientBoostingClassifier(n_estimators=300, max_depth=3, 
                                            learning_rate=0.03, subsample=0.8,
                                            min_samples_leaf=8, random_state=42)
        scores = cross_val_score(m, X_sorted_scaled, y_sorted, cv=tscv, scoring='neg_log_loss')
        print(f"  CV {name}: {-scores.mean():.4f} +/- {scores.std():.4f}")
    
    # Retrain best on ALL data
    final_scaler = StandardScaler()
    X_all_scaled = final_scaler.fit_transform(X)
    
    if best_name == 'LogReg':
        final_model = LogisticRegression(max_iter=1000, random_state=42)
    elif best_name == 'RF':
        final_model = RandomForestClassifier(n_estimators=300, max_depth=6, 
                                              min_samples_leaf=8, random_state=42)
    else:
        final_model = GradientBoostingClassifier(n_estimators=300, max_depth=3, 
                                                  learning_rate=0.03, subsample=0.8,
                                                  min_samples_leaf=8, random_state=42)
    final_model.fit(X_all_scaled, y)
    
    return final_model, final_scaler, best_name

# Train men's model
m_model, m_scaler, m_best = train_and_evaluate(m_tourney, m_diff_cols, "MEN")

# Train women's model  
w_model, w_scaler, w_best = train_and_evaluate(w_tourney, w_diff_cols, "WOMEN")

# Save models
joblib.dump(m_model, 'output/men_model.pkl')
joblib.dump(m_scaler, 'output/men_scaler.pkl')
joblib.dump(w_model, 'output/women_model.pkl')
joblib.dump(w_scaler, 'output/women_scaler.pkl')
print("\nModels saved to output/")


  MEN — Train: 2230, Test: 668
  Model        Log Loss   Accuracy
  LogReg         0.6042     0.6886
  RF             0.5853     0.6901
  GBM            0.6131     0.7081

  Best: RF (log_loss=0.5853)
  CV LogReg: 0.5662 +/- 0.0472
  CV RF: 0.5693 +/- 0.0360
  CV GBM: 0.5927 +/- 0.0354

  WOMEN — Train: 1260, Test: 662
  Model        Log Loss   Accuracy
  LogReg         0.4389     0.7825
  RF             0.4616     0.7779
  GBM            0.4926     0.7644

  Best: LogReg (log_loss=0.4389)
  CV LogReg: 0.4473 +/- 0.0395
  CV RF: 0.4505 +/- 0.0408
  CV GBM: 0.4896 +/- 0.0671

Models saved to output/


In [160]:
# ── Step 8: Generate submission predictions ──────────────────────────────

def generate_submission(sample_path, output_path, m_stats, w_stats, 
                        m_model, m_scaler, m_features, m_diff_cols,
                        w_model, w_scaler, w_features, w_diff_cols):
    """Generate predictions for all matchups in a submission file."""
    sub = pd.read_csv(sample_path)
    print(f"Loading {sample_path.name}: {len(sub)} matchups")
    
    # Parse ID -> Season, TeamLow, TeamHigh
    parts = sub['ID'].str.split('_', expand=True).astype(int)
    sub['Season'] = parts[0]
    sub['TeamLow'] = parts[1]
    sub['TeamHigh'] = parts[2]
    
    # Determine gender: 1xxx = Men, 3xxx = Women
    sub['Gender'] = np.where(sub['TeamLow'] < 2000, 'M', 'W')
    
    print(f"  Men matchups: {(sub['Gender']=='M').sum()}")
    print(f"  Women matchups: {(sub['Gender']=='W').sum()}")
    
    # Process men and women separately
    predictions = pd.Series(0.5, index=sub.index)  # default 0.5
    
    for gender, stats, model, scaler, features, diff_cols in [
        ('M', m_stats, m_model, m_scaler, m_features, m_diff_cols),
        ('W', w_stats, w_model, w_scaler, w_features, w_diff_cols),
    ]:
        mask = sub['Gender'] == gender
        if mask.sum() == 0:
            continue
            
        g_sub = sub[mask].copy()
        
        # Merge stats for TeamLow
        g_sub = g_sub.merge(
            stats, left_on=['Season', 'TeamLow'], right_on=['Season', 'TeamID'], how='left'
        )
        # Merge stats for TeamHigh
        g_sub = g_sub.merge(
            stats, left_on=['Season', 'TeamHigh'], right_on=['Season', 'TeamID'], 
            how='left', suffixes=('_Low', '_High')
        )
        
        # Compute difference features (Low - High)
        for f in features:
            g_sub[f'Diff_{f}'] = g_sub[f'{f}_Low'] - g_sub[f'{f}_High']
        
        # Check which rows have all features available
        has_features = g_sub[diff_cols].notna().all(axis=1)
        print(f"  {gender}: {has_features.sum()}/{len(g_sub)} matchups have features")
        
        if has_features.sum() > 0:
            X_pred = g_sub.loc[has_features, diff_cols].values
            X_pred_scaled = scaler.transform(X_pred)
            probs = model.predict_proba(X_pred_scaled)[:, 1]
            
            # Clip to avoid extreme log loss
            probs = np.clip(probs, 0.05, 0.95)
            
            # Map predictions back to original index
            predictions.loc[g_sub.index[has_features]] = probs
    
    sub['Pred'] = predictions.values
    
    # Save
    sub[['ID', 'Pred']].to_csv(output_path, index=False)
    print(f"\nSaved to {output_path}")
    print(f"  Predictions: min={sub['Pred'].min():.4f}, max={sub['Pred'].max():.4f}, "
          f"mean={sub['Pred'].mean():.4f}")
    print(f"  NaN count: {sub['Pred'].isnull().sum()}")
    print(f"  Rows at default 0.5: {(sub['Pred']==0.5).sum()}")
    
    return sub

# Generate Stage 1 (2022-2025)
print("="*60)
print("GENERATING STAGE 1 SUBMISSION")
print("="*60)
stage1 = generate_submission(
    DATA_DIR / "SampleSubmissionStage1.csv",
    Path("output/submission_stage1.csv"),
    m_stats, w_stats,
    m_model, m_scaler, MEN_FEATURES, m_diff_cols,
    w_model, w_scaler, WOMEN_FEATURES, w_diff_cols
)

print("\n" + "="*60)
print("GENERATING STAGE 2 SUBMISSION")
print("="*60)
# Generate Stage 2 (2026)
stage2 = generate_submission(
    DATA_DIR / "SampleSubmissionStage2.csv",
    Path("output/submission_stage2.csv"),
    m_stats, w_stats,
    m_model, m_scaler, MEN_FEATURES, m_diff_cols,
    w_model, w_scaler, WOMEN_FEATURES, w_diff_cols
)

GENERATING STAGE 1 SUBMISSION
Loading SampleSubmissionStage1.csv: 519144 matchups
  Men matchups: 261013
  Women matchups: 258131
  M: 261013/261013 matchups have features
  W: 258131/258131 matchups have features

Saved to output\submission_stage1.csv
  Predictions: min=0.0500, max=0.9500, mean=0.5028
  NaN count: 0
  Rows at default 0.5: 258131

GENERATING STAGE 2 SUBMISSION
Loading SampleSubmissionStage2.csv: 132133 matchups
  Men matchups: 66430
  Women matchups: 65703
  M: 66430/66430 matchups have features
  W: 65703/65703 matchups have features

Saved to output\submission_stage2.csv
  Predictions: min=0.0500, max=0.9500, mean=0.5087
  NaN count: 0
  Rows at default 0.5: 65703


In [161]:
# ── Step 9: Validation & sanity checks ───────────────────────────────────

def validate_submission(sub_df, sample_path):
    """Run sanity checks on a generated submission."""
    sample = pd.read_csv(sample_path)
    
    print(f"\nValidation for {sample_path.name}:")
    print(f"  Expected rows: {len(sample)}")
    print(f"  Generated rows: {len(sub_df)}")
    assert len(sub_df) == len(sample), "ROW COUNT MISMATCH!"
    print(f"  Row count: OK")
    
    # Check no NaN
    nan_count = sub_df['Pred'].isnull().sum()
    print(f"  NaN predictions: {nan_count}")
    assert nan_count == 0, "NaN predictions found!"
    
    # Check range
    print(f"  Pred range: [{sub_df['Pred'].min():.4f}, {sub_df['Pred'].max():.4f}]")
    assert sub_df['Pred'].min() >= 0.0, "Predictions below 0!"
    assert sub_df['Pred'].max() <= 1.0, "Predictions above 1!"
    
    # Check distribution
    print(f"  Pred mean: {sub_df['Pred'].mean():.4f} (should be ~0.5)")
    print(f"  Pred std:  {sub_df['Pred'].std():.4f}")
    print(f"  At default 0.5: {(sub_df['Pred']==0.5).sum()} ({100*(sub_df['Pred']==0.5).mean():.1f}%)")
    
    # IDs match exactly
    assert list(sub_df['ID']) == list(sample['ID']), "ID MISMATCH!"
    print(f"  IDs match: OK")
    
    print(f"  ALL CHECKS PASSED!")

validate_submission(stage1, DATA_DIR / "SampleSubmissionStage1.csv")
validate_submission(stage2, DATA_DIR / "SampleSubmissionStage2.csv")

# Spot-check: For 2025, compare predicted probabilities for known tournament matchups
print("\n" + "="*60)
print("SPOT CHECK: Prediction distribution by seed matchup (2026 men)")
print("="*60)
s2 = stage2.copy()
s2_men = s2[s2['Gender'] == 'M']
# Merge seed info for TeamLow and TeamHigh
s2_men = s2_men.merge(m_stats[m_stats['Season']==2026][['TeamID','SeedNum']], 
                       left_on='TeamLow', right_on='TeamID', how='left')
s2_men = s2_men.merge(m_stats[m_stats['Season']==2026][['TeamID','SeedNum']], 
                       left_on='TeamHigh', right_on='TeamID', how='left', suffixes=('_Low','_High'))

# Show average predicted win prob by seed difference bucket
s2_men['SeedDiff'] = s2_men['SeedNum_Low'] - s2_men['SeedNum_High']
s2_men['SeedDiffBucket'] = pd.cut(s2_men['SeedDiff'], bins=[-20,-10,-5,-2,2,5,10,20])
print(s2_men.groupby('SeedDiffBucket')['Pred'].agg(['mean','count']))


Validation for SampleSubmissionStage1.csv:
  Expected rows: 519144
  Generated rows: 519144
  Row count: OK
  NaN predictions: 0
  Pred range: [0.0500, 0.9500]
  Pred mean: 0.5028 (should be ~0.5)
  Pred std:  0.2255
  At default 0.5: 258131 (49.7%)
  IDs match: OK
  ALL CHECKS PASSED!

Validation for SampleSubmissionStage2.csv:
  Expected rows: 132133
  Generated rows: 132133
  Row count: OK
  NaN predictions: 0
  Pred range: [0.0500, 0.9500]
  Pred mean: 0.5087 (should be ~0.5)
  Pred std:  0.2386
  At default 0.5: 65703 (49.7%)
  IDs match: OK
  ALL CHECKS PASSED!

SPOT CHECK: Prediction distribution by seed matchup (2026 men)
                    mean  count
SeedDiffBucket                 
(-20, -10]      0.533928   2836
(-10, -5]       0.577928   7734
(-5, -2]        0.554935   6076
(-2, 2]         0.502863  32068
(2, 5]          0.507880   6827
(5, 10]         0.501024   8528
(10, 20]        0.484326   2361
